# 04: Contar, sumar y promediar: funciones de agregación



COUNT, SUM, AVG, MIN y MAX: Convertir miles de filas en un solo número que responde una pregunta de negocio.

## [01] Dashboard de métricas del e-commerce

El CEO quiere un resumen ejecutivo. 

Calcula en una sola consulta: 

- total de pedidos
- clientes activos (distintos)
- suma total de ventas (redondeada a 2 decimales)
- ticket medio (redondeado a 2 decimales)
- la fecha del último pedido. 

Solo cuenta pedidos completados.

Resultado esperado

| total_pedidos | clientes_activos | ventas_totales | ticket_medio | ultimo_pedido |
| ------------- | ---------------- | -------------- | ------------ | ------------- |
| 16199         | 3498             | 1680343.56     | 104.81       | 2024-12-30    |

```sql
SELECT
    COUNT(*) AS total_pedidos,
    COUNT(cliente_id) AS clientes_activos,
    SUM(importe) AS ventas_totales, 
    ROUND(AVG(importe), 2) AS ticket_medio,
    MAX(fecha) AS ultimo_pedido
FROM pedidos
WHERE estado = 'completado';
``` 

## [03] Rango de precios del catálogo


El equipo de pricing necesita un análisis del catálogo: 

1. producto más barato
2. más caro
3. precio medio
4. rango (max-min) 
5. total de productos. 

Todo en una query.

Resultado esperado

| total_productos | precio_minimo | precio_maximo | precio_medio | rango   |
| --------------- | ------------- | ------------- | ------------ | ------- |
| 200             | 6.24          | 1718.48       | 237.56       | 1712.24 |


```sql
SELECT
    COUNT(*) AS total_productos,
    MIN(precio) AS precio_minimo,
    MAX(precio) AS precio_maximo,
    ROUND(AVG(precio), 2) AS precio_medio,
    MAX(precio) - MIN(precio) AS rango
FROM productos;
```

## [04] Comparar estados de pedidos

Finanzas necesita desglosar pedidos por estado: 

para cada estado (completado, cancelado, devuelto): 

1. muestra cuántos pedidos hay
2. el importe total 
3. el ticket medio. 

Escribe una consulta por estado y apílalas con UNION ALL.

Resultado esperado

| estado     | pedidos | total      | ticket_medio |
| ---------- | ------- | ---------- | ------------ |
| cancelado  | 2373    | 247537.99  | 105.47       |
| completado | 16199   | 1680343.56 | 104.81       |
| devuelto   | 1190    | 121612.8   | 103.32       |

```sql
SELECT
    'completado' AS estado,
    COUNT(*) AS pedidos,
    ROUND(SUM(importe), 2) AS total,
    ROUND(AVG(importe), 2) AS ticket_medio
FROM pedidos
WHERE estado = 'completado';

UNION ALL

SELECT
    'cancelado' AS estado,
    COUNT(*) AS pedidos,
    ROUND(SUM(importe), 2) AS total,
    ROUND(AVG(importe), 2) AS ticket_medio
FROM pedidos
WHERE estado = 'cancelado';

UNION ALL

SELECT
    'devuelto' AS estado,
    COUNT(*) AS pedidos,
    ROUND(SUM(importe), 2) AS total,
    ROUND(AVG(importe), 2) AS ticket_medio
FROM pedidos
WHERE estado = 'devuelto';

```

## [05] Comparar primer y segundo semestre

El CFO quiere comparar las ventas completadas del primer semestre 2024 (ene-jun) vs segundo semestre 2024 (jul-dic). 

Para cada semestre: pedidos, total facturado y ticket medio.

Resultado esperado

| periodo | pedidos | total     | ticket_medio |
| ------- | ------- | --------- | ------------ |
| S1-2024 | 4790    | 497660.27 | 105.06       |
| S2-2024 | 2110    | 218851.9  | 104.71       |


```sql
SELECT 
    'S1-2024' AS periodo,
    COUNT(*) AS pedidos,
    ROUND(SUM(importe), 2) AS total,
    ROUND(AVG(importe), 2) AS ticket_medio
FROM pedidos
WHERE estado = 'completado'
    AND fecha >= '2024-01-01' AND fecha < '2024-07-01'

UNION ALL

SELECT 
    'S2-2024' AS periodo,
    COUNT(*) AS pedidos,
    ROUND(SUM(importe), 2) AS total,
    ROUND(AVG(importe), 2) AS ticket_medio
FROM pedidos
WHERE estado = 'completado'
    AND fecha >= '2024-07-01' AND fecha < '2024-12-31'
ORDER BY periodo
```